# Partie 2 -- Entrainement des Modeles de Boosting

## Prudential Life Insurance Assessment

**Objectif** : Entrainer XGBoost et CatBoost pour predire le niveau de risque (`Response`, 1 a 8).

---

### 4 configurations testees

| # | Modele | Methode |
|:--|:-------|:--------|
| 1 | XGBoost | Arrondi simple |
| 2 | XGBoost | Avec offset optimization |
| 3 | CatBoost | Arrondi simple |
| 4 | CatBoost | Avec offset optimization |

In [2]:
!pip install pandas numpy scikit-learn xgboost catboost scipy

  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl.metadata (11 kB)
  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
  Using cached catboost-1.2.10-cp310-cp310-win_amd64.whl.metadata (1.5 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached matplotlib-3.10.9-cp310-cp310-win_amd64.whl.metadata (52 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp310-cp310-win_amd64.whl.metadata (119 kB)
  Using cac


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostRegressor, CatBoostClassifier
from scipy.optimize import minimize_scalar
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
import warnings
import os

warnings.filterwarnings("ignore")
NUM_CLASSES = 8

---
## Fonctions utilitaires

In [4]:
def qwk(y_pred, y_true):
    """Quadratic Weighted Kappa."""
    y_true = np.array(y_true).astype(int)
    y_pred = np.array(y_pred)
    y_pred = np.clip(np.round(y_pred), np.min(y_true), np.max(y_true)).astype(int)
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")


def optimize_offsets(preds, labels):
    """
    Optimise un offset par classe pour maximiser le QWK.
    Utilise minimize_scalar (un seul parametre a la fois).
    """
    preds = np.array(preds, dtype=float)
    labels = np.array(labels, dtype=float)
    adjusted = preds.copy()
    offsets = np.zeros(NUM_CLASSES)
    
    for j in [6, 4, 5, 3, 2, 1, 7, 0]:
        def objective(x, cls=j):
            adjusted[preds.astype(int) == cls] = preds[preds.astype(int) == cls] + x
            return -qwk(adjusted, labels)
        
        result = minimize_scalar(objective, bounds=(-3, 3), method="bounded")
        offsets[j] = result.x
        adjusted[preds.astype(int) == j] = preds[preds.astype(int) == j] + offsets[j]
    
    return offsets


def apply_offsets(preds, offsets):
    """Applique les offsets et retourne les predictions finales (1-8)."""
    preds = np.array(preds, dtype=float)
    adjusted = preds.copy()
    for j in range(NUM_CLASSES):
        mask = preds.astype(int) == j
        adjusted[mask] = preds[mask] + offsets[j]
    return np.clip(np.round(adjusted), 1, 8).astype(int)


print("Fonctions definies.")

Fonctions definies.


---
## 1. Chargement des donnees

In [5]:
DATA_DIR = "prudential-life-insurance-assessment"
TARGET = "Response"

train = pd.read_csv(os.path.join(DATA_DIR, "train_clean.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test_clean.csv"))

feature_cols = [c for c in train.columns if c not in ["Id", TARGET]]
X = train[feature_cols]
y = train[TARGET].values
X_test = test[feature_cols]
test_ids = test["Id"].values


print(f"Train : {X.shape}")
print(f"Test  : {X_test.shape}")
print(f"Features : {len(feature_cols)}")

Train : (59381, 129)
Test  : (19765, 129)
Features : 129


---
## 2. Split train / validation (80/20)

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")

Train : (47504, 129)
Val   : (11877, 129)


---
## 3. Entrainement XGBoost

In [7]:
xgb_params = {
    "objective": "reg:squarederror",
    "eta": 0.05,
    "min_child_weight": 360,
    "subsample": 0.85,
    "colsample_bytree": 0.3,
    "max_depth": 7,
    "verbosity": 0,
}
XGB_ROUNDS = 720

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val, label=y_val)

print(f"Entrainement XGBoost ({XGB_ROUNDS} rounds)...")
xgb_model = xgb.train(
    xgb_params, dtrain, XGB_ROUNDS,
    evals=[(dtrain, "train"), (dval, "val")],
    verbose_eval=100,
)

xgb_val_preds = xgb_model.predict(dval)
xgb_train_preds = xgb_model.predict(dtrain)

# Score 1 : XGBoost arrondi simple
score_xgb_simple = qwk(xgb_val_preds, y_val)
print(f"\n[1] XGBoost arrondi simple  -> QWK = {score_xgb_simple:.4f}")

# Score 2 : XGBoost avec offsets
xgb_offsets = optimize_offsets(xgb_train_preds, y_train)
xgb_val_offset = apply_offsets(xgb_val_preds, xgb_offsets)
score_xgb_offset = qwk(xgb_val_offset, y_val)
print(f"[2] XGBoost avec offsets    -> QWK = {score_xgb_offset:.4f}  (gain : +{score_xgb_offset - score_xgb_simple:.4f})")

Entrainement XGBoost (720 rounds)...
[0]	train-rmse:2.42546	val-rmse:2.42669
[100]	train-rmse:1.86422	val-rmse:1.90519
[200]	train-rmse:1.82048	val-rmse:1.87510
[300]	train-rmse:1.79790	val-rmse:1.86703
[400]	train-rmse:1.78031	val-rmse:1.86428
[500]	train-rmse:1.76439	val-rmse:1.86314
[600]	train-rmse:1.74971	val-rmse:1.86250
[700]	train-rmse:1.73543	val-rmse:1.86226
[719]	train-rmse:1.73278	val-rmse:1.86234

[1] XGBoost arrondi simple  -> QWK = 0.5923
[2] XGBoost avec offsets    -> QWK = 0.6411  (gain : +0.0487)


---
## 4. Entrainement CatBoost

In [8]:
print("Entrainement CatBoost...")
cat_model = CatBoostRegressor(
    iterations=720,
    depth=7,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
    task_type="CPU",
)
cat_model.fit(X_train, y_train, eval_set=(X_val, y_val))

cat_val_preds = cat_model.predict(X_val)
cat_train_preds = cat_model.predict(X_train)

# Score 3 : CatBoost arrondi simple
score_cat_simple = qwk(cat_val_preds, y_val)
print(f"\n[3] CatBoost arrondi simple -> QWK = {score_cat_simple:.4f}")

# Score 4 : CatBoost avec offsets
cat_offsets = optimize_offsets(cat_train_preds, y_train)
cat_val_offset = apply_offsets(cat_val_preds, cat_offsets)
score_cat_offset = qwk(cat_val_offset, y_val)
print(f"[4] CatBoost avec offsets   -> QWK = {score_cat_offset:.4f}  (gain : +{score_cat_offset - score_cat_simple:.4f})")

Entrainement CatBoost...
0:	learn: 2.4204958	test: 2.4216196	best: 2.4216196 (0)	total: 153ms	remaining: 1m 49s
100:	learn: 1.8705870	test: 1.9010907	best: 1.9010907 (100)	total: 861ms	remaining: 5.28s
200:	learn: 1.8234881	test: 1.8663471	best: 1.8663471 (200)	total: 1.55s	remaining: 4.01s
300:	learn: 1.7910029	test: 1.8529718	best: 1.8529234 (299)	total: 2.31s	remaining: 3.21s
400:	learn: 1.7649588	test: 1.8471285	best: 1.8471132 (396)	total: 2.98s	remaining: 2.37s
500:	learn: 1.7430967	test: 1.8437535	best: 1.8437535 (500)	total: 3.65s	remaining: 1.59s
600:	learn: 1.7214934	test: 1.8415371	best: 1.8415068 (598)	total: 4.33s	remaining: 857ms
700:	learn: 1.7033500	test: 1.8402611	best: 1.8402434 (695)	total: 5s	remaining: 135ms
719:	learn: 1.6999888	test: 1.8403207	best: 1.8401906 (703)	total: 5.12s	remaining: 0us

bestTest = 1.840190566
bestIteration = 703

Shrink model to first 704 iterations.

[3] CatBoost arrondi simple -> QWK = 0.5994
[4] CatBoost avec offsets   -> QWK = 0.6532  

---
## 4. Entrainement CatBoost Classifier

In [10]:
target_enc_cols = [c for c in feature_cols if c.endswith("_target_enc")]
feature_cols_teacher = [c for c in feature_cols if c not in target_enc_cols]

print("Colonnes target_enc retirées du teacher :")
print(target_enc_cols)
print("Nombre features teacher :", len(feature_cols_teacher))

X_train_teacher = X_train[feature_cols_teacher]
X_val_teacher = X_val[feature_cols_teacher]

Colonnes target_enc retirées du teacher :
['Product_Info_3_target_enc', 'Employment_Info_2_target_enc', 'Medical_History_2_target_enc', 'Medical_History_1_target_enc', 'Medical_History_10_target_enc', 'Medical_History_15_target_enc', 'Medical_History_24_target_enc', 'Medical_History_32_target_enc']
Nombre features teacher : 121


In [11]:
def confidence_scores(proba):
    sorted_proba = np.sort(proba, axis=1)

    p_max = sorted_proba[:, -1]
    p_second = sorted_proba[:, -2]
    margin = p_max - p_second
    entropy = -np.sum(proba * np.log(proba + 1e-12), axis=1)
    pred = np.argmax(proba, axis=1) + 1

    return pred, p_max, margin, entropy


# =========================
# Teacher classifiers pour pseudo-labelling
# =========================

y_train_cls = y_train - 1
y_val_cls = y_val - 1

xgb_clf_params = {
    "objective": "multi:softprob",
    "num_class": 8,
    "eta": 0.05,
    "min_child_weight": 360,
    "subsample": 0.85,
    "colsample_bytree": 0.3,
    "max_depth": 7,
    "verbosity": 0,
}

dtrain_cls = xgb.DMatrix(X_train_teacher, label=y_train_cls)
dval_cls = xgb.DMatrix(X_val_teacher, label=y_val_cls)

print("Entrainement XGBoost classifier...")
xgb_clf = xgb.train(
    xgb_clf_params,
    dtrain_cls,
    num_boost_round=720,
    evals=[(dtrain_cls, "train"), (dval_cls, "val")],
    verbose_eval=100,
)

print("Entrainement CatBoost classifier...")
cat_clf = CatBoostClassifier(
    iterations=720,
    depth=7,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=100,
    task_type="CPU",
)

cat_clf.fit(X_train_teacher, y_train)
cat_val_proba = cat_clf.predict_proba(X_val_teacher)

xgb_val_proba = xgb_clf.predict(dval_cls)


xgb_pred, xgb_pmax, xgb_margin, xgb_entropy = confidence_scores(xgb_val_proba)
cat_pred, cat_pmax, cat_margin, cat_entropy = confidence_scores(cat_val_proba)

same_prediction = np.abs(xgb_pred - cat_pred) <= 1
mean_pmax = (xgb_pmax + cat_pmax) / 2
mean_margin = (xgb_margin + cat_margin) / 2

keep_val_mask = (
    same_prediction
    & (mean_pmax > 0.80)
    & (mean_margin > 0.25)
)

print("Validation pseudo-labelling")
print("Accord XGB/CatBoost :", same_prediction.mean())
print("Taux gardé :", keep_val_mask.mean())
print("QWK XGB classifier :", qwk(xgb_pred, y_val))
print("QWK CatBoost classifier :", qwk(cat_pred, y_val))

Entrainement XGBoost classifier...
[0]	train-mlogloss:1.79560	val-mlogloss:1.79597
[100]	train-mlogloss:1.36448	val-mlogloss:1.38768
[200]	train-mlogloss:1.30145	val-mlogloss:1.33631
[300]	train-mlogloss:1.27556	val-mlogloss:1.32057
[400]	train-mlogloss:1.25914	val-mlogloss:1.31332
[500]	train-mlogloss:1.24642	val-mlogloss:1.30897
[600]	train-mlogloss:1.23569	val-mlogloss:1.30626
[700]	train-mlogloss:1.22618	val-mlogloss:1.30439
[719]	train-mlogloss:1.22444	val-mlogloss:1.30427
Entrainement CatBoost classifier...
0:	learn: 1.9991320	total: 87.9ms	remaining: 1m 3s
100:	learn: 1.2959026	total: 7.91s	remaining: 48.5s
200:	learn: 1.2385576	total: 15.6s	remaining: 40.2s
300:	learn: 1.1946197	total: 23.2s	remaining: 32.3s
400:	learn: 1.1633729	total: 30.9s	remaining: 24.6s
500:	learn: 1.1369475	total: 38.7s	remaining: 16.9s
600:	learn: 1.1128870	total: 46.4s	remaining: 9.19s
700:	learn: 1.0912935	total: 54.2s	remaining: 1.47s
719:	learn: 1.0874102	total: 55.7s	remaining: 0us
Validation pseud

In [12]:
synthetic_path = os.path.join(DATA_DIR, "synthetic_unlabeled.csv")
synthetic = pd.read_csv(synthetic_path)

# Supprimer uniquement les colonnes en trop du synthétique
extra_cols = [c for c in synthetic.columns if c not in feature_cols_teacher]
print("Colonnes en trop supprimées :", extra_cols)

synthetic = synthetic.drop(columns=extra_cols)

# Vérifier qu'il ne manque rien
missing_cols = [c for c in feature_cols_teacher if c not in synthetic.columns]
print("Colonnes manquantes :", missing_cols)

if missing_cols:
    raise ValueError(f"Colonnes manquantes dans synthetic : {missing_cols}")

# IMPORTANT : garder exactement le même ordre que le train
synthetic_features = synthetic[feature_cols_teacher]

print("Nombre features teacher :", len(feature_cols_teacher))
print("Nombre features synthetic :", synthetic_features.shape[1])

X_syn = synthetic_features
dsyn = xgb.DMatrix(X_syn)

xgb_syn_proba = xgb_clf.predict(dsyn)
cat_syn_proba = cat_clf.predict_proba(X_syn)

xgb_syn_pred, xgb_syn_pmax, xgb_syn_margin, _ = confidence_scores(xgb_syn_proba)
cat_syn_pred, cat_syn_pmax, cat_syn_margin, _ = confidence_scores(cat_syn_proba)

same_prediction = xgb_syn_pred == cat_syn_pred
mean_pmax = (xgb_syn_pmax + cat_syn_pmax) / 2
mean_margin = (xgb_syn_margin + cat_syn_margin) / 2

pseudo_pred = xgb_syn_pred

threshold = np.full_like(mean_pmax, 0.55, dtype=float)

# Seuil plus strict pour les classes 5 et 8
threshold[np.isin(pseudo_pred, [5, 8])] = 0.75

keep_mask = (
    ((xgb_syn_pred == cat_syn_pred) | (np.abs(xgb_syn_pred - cat_syn_pred) <= 1))
    & (mean_pmax > threshold)
)

synthetic_filtered = synthetic.loc[keep_mask].copy()
synthetic_filtered["Response"] = pseudo_pred[keep_mask]
synthetic_filtered["confidence"] = mean_pmax[keep_mask]
synthetic_filtered["margin"] = mean_margin[keep_mask]
synthetic_filtered["threshold_used"] = threshold[keep_mask]

print("Données synthétiques initiales :", len(synthetic))
print("Données gardées :", len(synthetic_filtered))
print("Données rejetées :", len(synthetic) - len(synthetic_filtered))

print("\nRépartition des pseudo-labels gardés :")
print(
    synthetic_filtered["Response"]
    .value_counts()
    .sort_index()
)

print("\nPourcentages par classe :")
print(
    synthetic_filtered["Response"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

out_path = os.path.join(DATA_DIR, "synthetic_pseudo_labeled_filtered.csv")
synthetic_filtered.to_csv(out_path, index=False)

print("\nFichier sauvegardé :", out_path)

Colonnes en trop supprimées : ['Id', 'Product_Info_3', 'Employment_Info_2', 'Medical_History_1', 'Medical_History_2', 'Medical_History_10', 'Medical_History_15', 'Medical_History_24', 'Medical_History_32']
Colonnes manquantes : []
Nombre features teacher : 121
Nombre features synthetic : 121
Données synthétiques initiales : 300000
Données gardées : 25861
Données rejetées : 274139

Répartition des pseudo-labels gardés :
Response
1     1659
2     3286
4        4
5     1364
6     4815
7     3177
8    11556
Name: count, dtype: int64

Pourcentages par classe :
Response
1     6.42
2    12.71
4     0.02
5     5.27
6    18.62
7    12.28
8    44.69
Name: proportion, dtype: float64

Fichier sauvegardé : prudential-life-insurance-assessment\synthetic_pseudo_labeled_filtered.csv


In [13]:
print("Répartition XGBoost sans filtre :")
print(pd.Series(xgb_syn_pred).value_counts().sort_index())

print("\nRépartition CatBoost sans filtre :")
print(pd.Series(cat_syn_pred).value_counts().sort_index())

Répartition XGBoost sans filtre :
1    18956
2    24397
3       86
4     2010
5    42782
6    87712
7    35885
8    88172
Name: count, dtype: int64

Répartition CatBoost sans filtre :
1    36002
2    33981
3     2073
4     2745
5    27463
6    87666
7    35541
8    74529
Name: count, dtype: int64


In [14]:
MAX_PER_CLASS = 3000

synthetic_filtered_balanced = (
    synthetic_filtered
    .groupby("Response", group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), MAX_PER_CLASS), random_state=42))
    .reset_index(drop=True)
)

print("\nRépartition après limitation par classe :")
print(
    synthetic_filtered_balanced["Response"]
    .value_counts()
    .sort_index()
)

out_path = os.path.join(DATA_DIR, "synthetic_pseudo_labeled_filtered_balanced.csv")
synthetic_filtered_balanced.to_csv(out_path, index=False)

print("\nFichier sauvegardé :", out_path)


Répartition après limitation par classe :
Response
1    1659
2    3000
4       4
5    1364
6    3000
7    3000
8    3000
Name: count, dtype: int64

Fichier sauvegardé : prudential-life-insurance-assessment\synthetic_pseudo_labeled_filtered_balanced.csv


In [22]:
debug_df = pd.DataFrame({
    "xgb_pred": xgb_syn_pred,
    "cat_pred": cat_syn_pred,
    "same": xgb_syn_pred == cat_syn_pred,
    "mean_pmax": mean_pmax,
    "mean_margin": mean_margin,
})

print(debug_df.groupby("xgb_pred")[["mean_pmax", "mean_margin"]].mean())
print(debug_df.groupby("cat_pred")[["mean_pmax", "mean_margin"]].mean())

          mean_pmax  mean_margin
xgb_pred                        
1          0.413277     0.164860
2          0.450717     0.182368
3          0.298429     0.071963
4          0.328777     0.108515
5          0.486722     0.272241
6          0.407025     0.189294
7          0.425968     0.191537
8          0.580203     0.414782
          mean_pmax  mean_margin
cat_pred                        
1          0.418149     0.172480
2          0.439903     0.187942
3          0.417347     0.198461
4          0.320021     0.104211
5          0.538733     0.329734
6          0.406547     0.188593
7          0.426441     0.192381
8          0.598222     0.439877


---
## 5. Comparaison des 4 configurations

In [8]:
results = {
    "[1] XGBoost simple":  score_xgb_simple,
    "[2] XGBoost offset":  score_xgb_offset,
    "[3] CatBoost simple": score_cat_simple,
    "[4] CatBoost offset": score_cat_offset,
}

print(f"{'=' * 50}")
print(f"RESULTATS (Quadratic Weighted Kappa sur validation)")
print(f"{'=' * 50}")
for name, score in results.items():
    print(f"  {name:<25} : {score:.4f}")
print(f"{'─' * 50}")

best_name = max(results, key=results.get)
best_score = results[best_name]
print(f"  Meilleur : {best_name} (QWK = {best_score:.4f})")

RESULTATS (Quadratic Weighted Kappa sur validation)
  [1] XGBoost simple        : 0.5923
  [2] XGBoost offset        : 0.6411
  [3] CatBoost simple       : 0.5994
  [4] CatBoost offset       : 0.6532
──────────────────────────────────────────────────
  Meilleur : [4] CatBoost offset (QWK = 0.6532)


---
## 6. Soumission (meilleur modele sur le test)

On re-entraine le meilleur modele sur TOUT le train, puis on predit sur le test.

In [9]:
use_offsets = "offset" in best_name
use_catboost = "CatBoost" in best_name

print(f"Re-entrainement sur tout le train ({X.shape[0]} lignes)...")
print(f"Modele : {'CatBoost' if use_catboost else 'XGBoost'} | Offsets : {'oui' if use_offsets else 'non'}")

if use_catboost:
    final_model = CatBoostRegressor(
        iterations=720, depth=7, learning_rate=0.05,
        loss_function="RMSE", random_seed=42, verbose=0, task_type="CPU",
    )
    final_model.fit(X, y)
    full_preds = final_model.predict(X)
    test_preds_raw = final_model.predict(X_test)
else:
    dfull = xgb.DMatrix(X, label=y)
    dtest = xgb.DMatrix(X_test)
    final_model = xgb.train(xgb_params, dfull, XGB_ROUNDS, verbose_eval=False)
    full_preds = final_model.predict(dfull)
    test_preds_raw = final_model.predict(dtest)

if use_offsets:
    final_offsets = optimize_offsets(full_preds, y)
    final_predictions = apply_offsets(test_preds_raw, final_offsets)
else:
    final_predictions = np.clip(np.round(test_preds_raw), 1, 8).astype(int)

print(f"\nPredictions generees : {len(final_predictions)}")
print(f"\nDistribution :")
print(pd.Series(final_predictions).value_counts().sort_index())

Re-entrainement sur tout le train (59381 lignes)...
Modele : CatBoost | Offsets : oui

Predictions generees : 19765

Distribution :
1    1508
2    1412
3    1107
4    2164
5    2019
6    1942
7    3406
8    6207
Name: count, dtype: int64


In [10]:
submission = pd.DataFrame({"Id": test_ids, "Response": final_predictions})
submission_path = os.path.join(DATA_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)

print(f"Fichier : {submission_path}")
print(f"Format  : {submission.shape}")
submission.head(10)

Fichier : prudential-life-insurance-assessment\submission.csv
Format  : (19765, 2)


,Id,Response
0,1,2
1,3,6
2,4,7
3,9,7
4,12,7
5,13,8
6,21,7
7,28,8
8,30,4
9,36,8


---
## Conclusion

| # | Configuration | QWK (validation) |
|:--|:-------------|:-----------------|
| 1 | XGBoost arrondi simple | voir ci-dessus |
| 2 | XGBoost + offsets | voir ci-dessus |
| 3 | CatBoost arrondi simple | voir ci-dessus |
| 4 | CatBoost + offsets | voir ci-dessus |

Le meilleur modele est utilise pour generer `submission.csv`.